# Training-side free generation after the preference pilot
INFERENCE ONLY: eleven fixed training paragraphs x two saved epoch-2 adapters = 22 new generations. Zero optimizer steps. Eleven exact archived control responses are reused in local analysis.
Create a NEW PRIVATE notebook. Attach only the two upload files from data/preference_training_inference_v1/upload/ as a private Dataset. Do not upload private_key/.
Attach notebook OUTPUTS of oliverlds/preference-pair-pilot-4b version 1, including sft/epoch_2/adapter and dpo/epoch_2/adapter. No original expanded-pilot adapter needed.
Select GPU T4 x2; only GPU 0 is exposed. Enable Internet and HF_TOKEN in Secrets before Save & Run All. Keep credentials outside code.
Use a fresh session with Torch 2.11.0+cu128. All model/package/adapter hashes and exact training prompt tokens are checked. No truncation or prompt changes.
The upload contains source prompts only, with no targets or archived outputs. All parameters are frozen; there is no training or checkpoint saving.
The thirty-minute cap includes setup. Download preference_training_inference_v1/ and the notebook log after completion, including partial outputs on failure.
This is a seen-training diagnostic, not held-out evaluation or evidence of Fama style/human authorship. No detector calls or automatic submission.


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
NOTEBOOK_STARTED=time.monotonic()
if 'torch' in sys.modules:raise RuntimeError('Restart a fresh session and execute this cell first')
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['DO_NOT_TRACK']='1'
def remaining():
    value=1800-(time.monotonic()-NOTEBOOK_STARTED)
    if value<1:raise TimeoutError('Thirty-minute inference budget exhausted')
    return value
dependencies=['transformers==5.20.1','peft==0.21.2','accelerate==1.15.0',
              'bitsandbytes==0.50.2','huggingface-hub==1.33.0','sentencepiece','Pillow','jinja2==3.1.6']
subprocess.run([sys.executable,'-m','pip','install','-q','-U']+dependencies,check=True,timeout=remaining())


In [ ]:
from pathlib import Path
(Path('/kaggle/working')/'preference_training_inference.py').write_text('"""Frozen-adapter, training-side free generation: no optimization or target prompts."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport time\n\nPROTOCOL=\'preference_training_inference_v1\'\nMODEL_ID=\'google/gemma-3-4b-it\'\nREVISION=\'093f9f388b31de276ce2de164bdc2081324b9767\'\nINSTRUCTION=\'\'\'Rewrite the passage as one paragraph of natural academic economics prose.\nRetain its substantive claims, actors, quantities, comparisons, conditions,\nexceptions, uncertainty, negation, attribution and causal direction. Improve\nexpression and organization where useful, without adding economic facts or\nrelationships. Do not fill gaps using outside knowledge. Return only the paragraph.\'\'\'\nDECODING={\'do_sample\':False,\'num_beams\':1,\'max_new_tokens\':512,\'repetition_penalty\':1.0}\nARMS=(\'sft\',\'dpo\')\nFIELDS={\'pair_id\',\'anchor_group_id\',\'example_id\',\'paper_id\',\'input_kind\',\'family\',\'source\',\'prompt\',\n        \'source_sha256\',\'prompt_sha256\',\'prefix_ids\',\'prefix_ids_sha256\'}\n\n\ndef sha(text):return hashlib.sha256(text.encode()).hexdigest()\ndef file_sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\ndef write_json(path,value):path.write_text(json.dumps(value,indent=2,ensure_ascii=False)+\'\\n\')\ndef prompt(text):return INSTRUCTION+\'\\n\\nPassage:\\n\'+text\ndef messages(job):return [{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}]\n\n\ndef select_rows(rows):\n    """Prefer an included original input, otherwise its included variant; no output ranking."""\n    groups={}\n    for row in rows:groups.setdefault(row[\'anchor_group_id\'],[]).append(row)\n    return [sorted(group,key=lambda r:(r[\'input_kind\']!=\'original\',r[\'pair_id\']))[0]\n            for _,group in sorted(groups.items())]\n\n\ndef load_inputs(jobs_path,config_path):\n    config=json.loads(config_path.read_text())\n    if (config.get(\'protocol\')!=PROTOCOL or config.get(\'model_id\')!=MODEL_ID or config.get(\'model_revision\')!=REVISION\n            or config.get(\'decoding\')!=DECODING or config.get(\'arms\')!=list(ARMS) or config.get(\'seed\')!=1729\n            or config.get(\'max_length\')!=1024 or config.get(\'maximum_runtime_seconds\')!=1800\n            or config.get(\'training_authorized\') is not False or config.get(\'optimizer_steps\')!=0\n            or config.get(\'jobs_sha256\')!=file_sha(jobs_path) or config.get(\'runtime_source_sha256\')!=file_sha(Path(__file__))):\n        raise ValueError(\'frozen_inference_configuration_changed\')\n    jobs=[json.loads(line) for line in jobs_path.read_text().splitlines() if line.strip()]\n    if (len(jobs)!=11 or len({r[\'pair_id\'] for r in jobs})!=11 or len({r[\'anchor_group_id\'] for r in jobs})!=11\n            or len({r[\'example_id\'] for r in jobs})!=11 or len({r[\'paper_id\'] for r in jobs})!=10\n            or [r[\'pair_id\'] for r in jobs]!=config[\'pair_ids\']\n            or Counter(r[\'input_kind\'] for r in jobs)!=Counter({\'original\':9,\'variant\':2})):\n        raise ValueError(\'frozen_eleven_group_selection_changed\')\n    for row in jobs:\n        ids=row[\'prefix_ids\']\n        if (set(row)!=FIELDS or not row[\'source\'].strip() or row[\'prompt\']!=prompt(row[\'source\'])\n                or sha(row[\'source\'])!=row[\'source_sha256\'] or sha(row[\'prompt\'])!=row[\'prompt_sha256\']\n                or not isinstance(ids,list) or not ids or any(type(i) is not int or i<0 for i in ids)\n                or ids[0]!=2 or ids.count(2)!=1 or len(ids)+512>1024\n                or sha(json.dumps(ids))!=row[\'prefix_ids_sha256\']):raise ValueError(\'source_only_prompt_or_prefix_changed\')\n    if set(config[\'adapter_sha256\'])!=set(ARMS) or any(set(v)!={\'adapter_config.json\',\'adapter_model.safetensors\'}\n            for v in config[\'adapter_sha256\'].values()):raise ValueError(\'two_exact_epoch_two_adapters_required\')\n    return jobs,config\n\n\ndef verify_adapter(folder,expected):\n    if any(file_sha(folder/name)!=digest for name,digest in expected.items()):raise ValueError(\'wrong_saved_pilot_adapter\')\n    c=json.loads((folder/\'adapter_config.json\').read_text())\n    if (c[\'base_model_name_or_path\']!=MODEL_ID or c[\'r\']!=8 or c[\'lora_alpha\']!=16 or c[\'lora_dropout\']!=0.05):\n        raise ValueError(\'saved_adapter_configuration_changed\')\n\n\ndef run_training(jobs,config,adapters,output,max_seconds=1800):\n    """Repository lazy-ML entrypoint convention; performs inference only."""\n    if output.exists():raise ValueError(\'fresh_inference_output_required\')\n    for arm in ARMS:verify_adapter(adapters[arm],config[\'adapter_sha256\'][arm])\n    output.mkdir(parents=True);started=time.monotonic()\n    state={\'complete\':False,\'phase\':\'start\',\'training_called\':False,\'optimizer_steps\':0,\'completed_generations\':0}\n    report={\'protocol\':PROTOCOL,\'mode\':\'inference_only\',\'complete\':False,\'config\':config,\n        \'training_called\':False,\'optimizer_steps\':0,\'snapshots\':[]}\n    def persist():write_json(output/\'state.json\',state);write_json(output/\'evaluation.json\',report)\n    def budget():\n        if time.monotonic()-started>=min(max_seconds,1800):raise TimeoutError(\'inference_time_budget_exceeded\')\n    def phase(name):state[\'phase\']=name;persist();print(\'Phase: \'+name,flush=True);budget()\n    persist()\n    try:\n        import torch\n        from huggingface_hub import HfApi\n        from transformers import AutoProcessor,Gemma3ForConditionalGeneration,BitsAndBytesConfig,GenerationConfig,LogitsProcessor,LogitsProcessorList\n        from peft import PeftModel\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1 or \'T4\' not in torch.cuda.get_device_name(0):\n            raise RuntimeError(\'Expose only GPU 0 on Kaggle T4 x2\')\n        packages={p:importlib.metadata.version(p) for p in config[\'packages\']}\n        if packages!=config[\'packages\']:raise RuntimeError(\'recorded_pilot_environment_required\')\n        torch.cuda.set_device(0);torch.manual_seed(1729);torch.cuda.reset_peak_memory_stats(0)\n        report[\'runtime\']={\'packages\':packages,\'model_id\':MODEL_ID,\'model_revision\':REVISION,\n            \'gpu\':torch.cuda.get_device_name(0),\'gpu_count\':1,\'runtime_source_sha256\':file_sha(Path(__file__))}\n        phase(\'model_access\')\n        try:\n            from kaggle_secrets import UserSecretsClient\n            token=UserSecretsClient().get_secret(\'HF_TOKEN\')\n        except Exception:raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\') from None\n        if not isinstance(token,str) or not token.strip():raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\')\n        if HfApi().model_info(MODEL_ID,revision=REVISION,token=token,timeout=60).sha!=REVISION:raise RuntimeError(\'model_revision_changed\')\n        processor=AutoProcessor.from_pretrained(MODEL_ID,revision=REVISION,token=token);tokenizer=processor.tokenizer\n        phase(\'prompt_equivalence\');audits=[]\n        for job in jobs:\n            values=processor.apply_chat_template(messages(job),tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors=\'pt\')\n            ids=values[\'input_ids\'][0].tolist()\n            if ids!=job[\'prefix_ids\'] or (\'token_type_ids\' in values and bool(values[\'token_type_ids\'].any())):\n                raise ValueError(\'processor_prefix_differs_from_verified_training_ids\')\n            audits.append({\'pair_id\':job[\'pair_id\'],\'prompt_tokens\':len(ids),\'prefix_ids_sha256\':sha(json.dumps(ids)),\'matches_training_prefix\':True})\n        write_json(output/\'token_audit.json\',audits)\n        phase(\'load_model\')\n        base=Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=REVISION,token=token,dtype=torch.float32,\n            device_map={\'\':0},attn_implementation=\'sdpa\',quantization_config=BitsAndBytesConfig(load_in_4bit=True,\n                bnb_4bit_quant_type=\'nf4\',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32))\n        token=None;base.eval();base.requires_grad_(False)\n        generation=GenerationConfig(**DECODING,eos_token_id=base.generation_config.eos_token_id,\n            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,use_cache=True)\n        if generation.to_dict()!=config[\'expected_generation_config\']:raise ValueError(\'recorded_decoding_configuration_changed\')\n        write_json(output/\'generation_config.json\',generation.to_dict())\n        class FiniteScores(LogitsProcessor):\n            def __call__(self,input_ids,scores):\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):\n                    raise RuntimeError(\'nonfinite_generation_scores\')\n                return scores\n        for arm in ARMS:\n            phase(\'load_\'+arm);torch.manual_seed(1729)\n            model=PeftModel.from_pretrained(base,str(adapters[arm]),is_trainable=False)\n            model.eval();model.requires_grad_(False)\n            if any(p.requires_grad for p in model.parameters()):raise RuntimeError(\'all_parameters_must_be_frozen\')\n            snapshot={\'arm\':arm,\'checkpoint_epoch\':2,\'complete\':False,\'all_parameters_frozen\':True,\'generations\':[]}\n            report[\'snapshots\'].append(snapshot);phase(\'generate_\'+arm)\n            for job in jobs:\n                budget();state[\'current_pair_id\']=job[\'pair_id\'];persist()\n                ids=torch.tensor([job[\'prefix_ids\']],dtype=torch.long,device=\'cuda:0\')\n                with torch.inference_mode():result=model.generate(input_ids=ids,attention_mask=torch.ones_like(ids),\n                    token_type_ids=torch.zeros_like(ids),generation_config=generation,logits_processor=LogitsProcessorList([FiniteScores()]))\n                new=result[0,ids.shape[-1]:];text=tokenizer.decode(new,skip_special_tokens=True)\n                eos=generation.eos_token_id;eos=eos if isinstance(eos,list) else [eos]\n                snapshot[\'generations\'].append({k:job[k] for k in (\'pair_id\',\'example_id\',\'anchor_group_id\',\'paper_id\',\'input_kind\',\'source_sha256\',\'prompt_sha256\')}|\n                    {\'output\':text,\'output_sha256\':sha(text),\'output_tokens\':int(new.numel()),\'empty\':not bool(text.strip()),\n                     \'output_limited\':bool(new.numel()>=512 and int(new[-1]) not in eos),\n                     \'copied_source\':\' \'.join(text.split())==\' \'.join(job[\'source\'].split())})\n                state[\'completed_generations\']+=1;persist();print(arm+\': saved %d/11 generations\'%len(snapshot[\'generations\']),flush=True)\n                del ids,result,new;budget()\n            snapshot[\'complete\']=True;persist()\n            base=model.unload();del model;base.eval();base.requires_grad_(False)\n        if state[\'completed_generations\']!=22:raise ValueError(\'fixed_generation_coverage_failed\')\n        for arm in ARMS:verify_adapter(adapters[arm],config[\'adapter_sha256\'][arm])\n        report[\'complete\']=True\n        state.update(complete=True,phase=\'complete\',elapsed_seconds=time.monotonic()-started,\n            peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated(0));persist()\n    except Exception as error:\n        state.update(failed_phase=state[\'phase\'],phase=\'failed\',error_type=type(error).__name__);persist()\n        raise RuntimeError(\'Inference stopped; preserve state and partial outputs. Error type: \'+type(error).__name__) from None\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--jobs\',type=Path,required=True);p.add_argument(\'--config\',type=Path,required=True)\n    p.add_argument(\'--sft-adapter\',type=Path);p.add_argument(\'--dpo-adapter\',type=Path);p.add_argument(\'--check\',action=\'store_true\')\n    p.add_argument(\'--run-inference\',action=\'store_true\');p.add_argument(\'--max-runtime-seconds\',type=int,default=1800)\n    p.add_argument(\'--output\',type=Path,default=Path(\'/kaggle/working/preference_training_inference_v1\'));a=p.parse_args(argv)\n    jobs,config=load_inputs(a.jobs,a.config);adapters={\'sft\':a.sft_adapter,\'dpo\':a.dpo_adapter}\n    for arm,path in adapters.items():\n        if path is not None:verify_adapter(path,config[\'adapter_sha256\'][arm])\n    if a.check:print(\'Validated eleven training prompts, two frozen adapters, 22 new generations. No model or training.\');return\n    if not a.run_inference or any(p is None for p in adapters.values()) or not 0<a.max_runtime_seconds<=1800:\n        raise ValueError(\'explicit_inference_flag_two_adapters_and_bounded_runtime_required\')\n    run_training(jobs,config,adapters,a.output,a.max_runtime_seconds)\n\n\nif __name__==\'__main__\':main()\n')


In [ ]:
import hashlib
import json
inputs=Path('/kaggle/input')
jobs=list(inputs.rglob('preference_training_jobs.jsonl'))
configs=list(inputs.rglob('preference_training_config.json'))
if len(jobs)!=1 or len(configs)!=1:raise RuntimeError('Attach exactly one new private training-inference Dataset with its two upload files')
expected={'jobs_sha256': 'd2434223a78f2162baa9dbc88c8a5283aa51da581abd6e56fe626c8f5b70d2c0', 'config_sha256': '8e652ae6e67057b5c4ec9f2a5e0007853dd3f608a9543b608b8de1c8ae12b779'}
for path,key in ((jobs[0],'jobs_sha256'),(configs[0],'config_sha256')):
    if hashlib.sha256(path.read_bytes()).hexdigest()!=expected[key]:raise RuntimeError('Frozen inference payload changed')
config=json.loads(configs[0].read_text())
command=[sys.executable,'/kaggle/working/preference_training_inference.py','--jobs',str(jobs[0]),'--config',str(configs[0])]
for arm in ('sft','dpo'):
    matches=[]
    for weights in inputs.rglob('adapter_model.safetensors'):
        folder=weights.parent
        if all((folder/name).is_file() and hashlib.sha256((folder/name).read_bytes()).hexdigest()==digest
               for name,digest in config['adapter_sha256'][arm].items()):matches.append(folder)
    if len(matches)!=1:raise RuntimeError('Attach outputs of oliverlds/preference-pair-pilot-4b version 1, including both final epoch_2 adapters')
    command+=['--'+arm+'-adapter',str(matches[0])]
subprocess.run(command+['--check'],check=True,timeout=remaining())
subprocess.run(command+['--run-inference','--max-runtime-seconds',str(int(remaining())),
    '--output','/kaggle/working/preference_training_inference_v1'],check=True,timeout=remaining())
